In [1]:

# Creación de Entorno

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

# Crear la sesión de Spark con el nombre "JJOO Analysis"
spark = SparkSession.builder \
    .appName("JJOO Analysis") \
    .getOrCreate()

# Crear Spark context
sc = spark.sparkContext

spark

c:\Users\Nicolas Soletic\mi_app2\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [2]:
import csv
from pyspark.sql import Row

# 1. Crear RDD deportista con 6 particiones
rdd_deportista = sc.textFile("../data/deportista.csv", minPartitions=6)

# 2. Crear RDD deportista2
rdd_deportista2 = sc.textFile("../data/deportista2.csv")

# Verificar particiones del primer RDD
print("Particiones deportista:", rdd_deportista.getNumPartitions())

# 3. Crear RDD deportistaTotal con la unión de ambos RDDs
deportistaTotal = rdd_deportista.union(rdd_deportista2)

# 4. Mostrar cantidad de registros contenidos en deportistaTotal
print("Cantidad de registros:", deportistaTotal.count())

# Función para parsear CSV correctamente
def parse_csv(line):
    return next(csv.reader([line]))

# Separar columnas usando csv.reader
deportista_parseado = deportistaTotal.map(parse_csv)

# Dejar solo registros que tengan 7 columnas
deportista_parseado = deportista_parseado.filter(lambda x: len(x) == 7)

# 5. Convertir el RDD deportistaTotal en un DataFrame llamado deportista
deportista = deportista_parseado.toDF([
    "deportista_id",
    "nombre",
    "genero",
    "edad",
    "altura",
    "peso",
    "equipo_id"
])

# Registros y estructura del DataFrame deportista
deportista.show(10)

deportista.printSchema()

Particiones deportista: 6
Cantidad de registros: 135571
+-------------+--------------------+------+----+------+----+---------+
|deportista_id|              nombre|genero|edad|altura|peso|equipo_id|
+-------------+--------------------+------+----+------+----+---------+
|            1|           A Dijiang|     1|  24|   180|  80|      199|
|            2|            A Lamusi|     1|  23|   170|  60|      199|
|            3| Gunnar Nielsen Aaby|     1|  24|     0|   0|      273|
|            4|Edgar Lindenau Aabye|     1|  34|     0|   0|      278|
|            5|Christine Jacoba ...|     2|  21|   185|  82|      705|
|            6|     Per Knut Aaland|     1|  31|   188|  75|     1096|
|            7|        John Aalberg|     1|  31|   183|  72|     1096|
|            8|Cornelia Cor Aalt...|     2|  18|   168|   0|      705|
|            9|    Antti Sami Aalto|     1|  26|   186|  96|      350|
|           10|Einar Ferdinand E...|     1|  26|     0|   0|      350|
+-------------+------

In [3]:
# Separar cada línea por coma
deportistaRDD = deportistaTotal.map(lambda x: x.split(","))

# 1. Crear RDD MayorEdad: deportistas mayores de edad, se asume mayor de 18
MayorEdad = deportistaRDD.filter(lambda x: int(x[3]) >= 18)

print("Primeros deportistas mayores de edad:")
for registro in MayorEdad.take(10):
    print(registro)


# 2. Crear RDD Deportistas_mujer: género femenino, sexo = 2
Deportistas_mujer = deportistaRDD.filter(lambda x: x[2] == "2")

print("Primeras deportistas mujeres:")
for registro in Deportistas_mujer.take(10):
    print(registro)


# 3. Convertir a mayúsculas todas las palabras del RDD deportistaTotal
deportistaTotal_mayuscula = deportistaTotal.map(lambda x: x.upper())

print("Primeros registros en mayúsculas:")
for registro in deportistaTotal_mayuscula.take(10):
    print(registro)

Primeros deportistas mayores de edad:
['1', 'A Dijiang', '1', '24', '180', '80', '199']
['2', 'A Lamusi', '1', '23', '170', '60', '199']
['3', 'Gunnar Nielsen Aaby', '1', '24', '0', '0', '273']
['4', 'Edgar Lindenau Aabye', '1', '34', '0', '0', '278']
['5', 'Christine Jacoba Aaftink', '2', '21', '185', '82', '705']
['6', 'Per Knut Aaland', '1', '31', '188', '75', '1096']
['7', 'John Aalberg', '1', '31', '183', '72', '1096']
['8', 'Cornelia Cor Aalten Strannood ', '2', '18', '168', '0', '705']
['9', 'Antti Sami Aalto', '1', '26', '186', '96', '350']
['10', 'Einar Ferdinand Einari Aalto', '1', '26', '0', '0', '350']
Primeras deportistas mujeres:
['5', 'Christine Jacoba Aaftink', '2', '21', '185', '82', '705']
['8', 'Cornelia Cor Aalten Strannood ', '2', '18', '168', '0', '705']
['13', 'Minna Maarit Aalto', '2', '30', '159', '55.5', '350']
['14', 'Pirjo Hannele Aalto Mattila ', '2', '32', '171', '65', '350']
['21', 'Ragnhild Margrethe Aamodt', '2', '27', '163', '0', '742']
['22', 'Andreea

In [10]:
# Generación de DataFrames Evento, Resultado, Equipos y Juego 

# Eventos
evento = spark.read.csv(
    "../data/evento.csv",
    header=True,
    inferSchema=True
)

# Equipos
equipos = spark.read.csv(
    "../data/equipo.csv",
    header=True,
    inferSchema=True
)

# Resultados
resultado = spark.read.csv(
    "../data/resultados.csv",
    header=True,
    inferSchema=True
)

# Juego
juego = spark.read.option("multiLine", "true").json("../data/juegos.json")


In [11]:
#Optimización de los DataFrames creados

evento.cache()
equipos.cache()
resultado.cache()
juego.cache()
deportista.cache()

evento.count()
equipos.count()
resultado.count()
juego.count()
deportista.count()

135570